# Understanding LLM Context

> **Description:** This notebook builds up the idea of *context* one layer at a time — starting from a single stateless API call and ending with the techniques real production systems use to manage it.

Large Language Models have no memory, no database, and no built-in sense of "the conversation so far." Every single API call is stateless — the model only knows what's inside the `messages` list you send it *this time*. That list — system instructions, conversation history, retrieved documents, tool results, few-shot examples — **is the context**, and it is the single most powerful lever you have as an AI engineer. Prompting, RAG, memory, agents, and cost control are, underneath it all, different flavors of one question: *what do I put in the context, and how do I keep it small enough and relevant enough to work?*

## What You'll Learn

| # | Section |
|---|---|
| 1 | A baseline call with the raw OpenAI SDK |
| 2 | Anatomy of the context window |
| 3 | Steering behavior with a system prompt |
| 4 | Giving the model memory: conversation history |
| 5 | Context has a limit: tokens & truncation |
| 6 | Grounding: injecting knowledge the model doesn't have |
| 7 | Few-shot examples: teaching by showing |
| 8 | Compacting context: summarizing instead of truncating |



## Setup

Load `OPENAI_API_KEY` from `.env` and create a single reusable OpenAI client.

> ⚠️ **Run this notebook from the `part_2_concepts/` folder** — that's what makes the relative `.env` path work, matching the other notebooks here.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(".env", override=True)
print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

OPENAI_API_KEY set: True


## 1. Baseline: A Plain Call with the OpenAI SDK

The rawest possible LLM call: one `user` message, no system prompt, no history, no tools. This is the [Chat Completions API](https://platform.openai.com/docs/api-reference/chat) — the foundation every higher-level SDK wraps.

In [2]:
from openai import OpenAI

client = OpenAI()
model = "gpt-5.4-nano"

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "user", "content": "What is the capital of France, and share one fun fact about it?"}
    ],
)
print(response.choices[0].message.content)

The capital of France is **Paris**.  

**Fun fact:** Paris is home to **the Louvre**, which was originally built as a royal palace and is now the world’s largest art museum.


👉 Now ask a follow-up **in a brand-new call**, without resending anything:

In [3]:
follow_up = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": "What city did I just ask you about?"}],
)
print(follow_up.choices[0].message.content)

I don’t have enough context to know which city you mean—please tell me what you asked earlier (or paste the question), and I’ll help.


👉 The model has no idea. There is no hidden session, no memory, no "last conversation" — the API is completely **stateless**. Every call only knows what's inside its own `messages` list. That list is the entire universe the model can reason about, and building it deliberately is what the rest of this notebook is about: **context engineering**.

## 2. Anatomy of the Context Window

The `messages` list you send *is* the context. Every call ships the **entire** conversation over the wire — nothing is stored server-side for you to draw on later (that's what memory, RAG, and summarization exist to fake). Each message has a `role`:

| Role | Meaning |
|---|---|
| `system` | Standing instructions — persona, rules, constraints |
| `user` | What the human (or your app) said |
| `assistant` | What the model said previously |
| `tool` | Results your code fed back from a function call |

Context is measured in **tokens**, not characters or words — roughly ¾ of a word each in English. Tokens are the actual unit models are priced, billed, and capacity-limited by, so let's count them with [`tiktoken`](https://github.com/openai/tiktoken), the tokenizer OpenAI models use.

In [4]:
import tiktoken

encoding = tiktoken.get_encoding("o200k_base")


def count_tokens(messages):
    return sum(len(encoding.encode(m["content"])) for m in messages if m.get("content"))


messages = [
    {"role": "system", "content": "You are a concise, friendly assistant."},
    {"role": "user", "content": "What is the capital of France, and share one fun fact about it?"},
]

print("The context sent to the model:")
for m in messages:
    print(f"  [{m['role']}] {m['content']}")

print("\nApprox. tokens in this request:", count_tokens(messages))

The context sent to the model:
  [system] You are a concise, friendly assistant.
  [user] What is the capital of France, and share one fun fact about it?

Approx. tokens in this request: 23


## 3. Steering Behavior with a System Prompt

The `system` message doesn't add facts — it adds **standing instructions** that color every response. Same question, same model, only the context changes:

In [6]:
question = "Tell me a software engineering joke"

# plain = client.chat.completions.create(
#     model=model,
#     messages=[{"role": "user", "content": question}],
# )
# print("No system prompt:\n", plain.choices[0].message.content)

pirate = client.chat.completions.create(
    model=model,
    messages=[
        { "role": "system", "content": ("You are a grizzled pirate captain. Answer every question in heavy pirate slang."), },
        { "role": "user", "content": question },
    ],
)
print("\nWith a pirate system prompt:\n", pirate.choices[0].message.content)


With a pirate system prompt:
 Arrr, matey—here be one fer ye:

Why did the bug walk the plank?  
’Cause it couldn’t be *patched*, and it kept sailin’ back into the same ol’ code!


👉 Nothing about Python changed. Only the **context** did — and it fully controlled tone, persona, and framing while the underlying advice stayed correct.

## 4. Giving the Model Memory: Conversation History

Section 1 showed the model forgets everything between calls. The fix is not magic — it's discipline: **you** keep a running `messages` list and resend it, growing it by one exchange per turn.

In [7]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "My name is Rahul and my favorite language is Python."},
]

response = client.chat.completions.create(model=model, messages=messages)
reply = response.choices[0].message.content
print("Assistant:", reply)

# Append both sides of the exchange before the next turn
messages.append({"role": "assistant", "content": reply})
messages.append({"role": "user", "content": "What's my name, and what language do I like?"})

response = client.chat.completions.create(model=model, messages=messages)
print("\nAssistant:", response.choices[0].message.content)

Assistant: Hi Rahul! Nice to meet you 😊  
Python is a great choice—what do you like most about it (data science, web apps, automation, scripting, etc.)?

Assistant: Your name is **Rahul**, and your favorite language is **Python**.


👉 This time it remembers — because "Rahul" and "Python" are still sitting in the `messages` list you sent. **Memory is just context you chose to keep and resend.** There's no session on OpenAI's servers doing this for you.

## 5. Context Has a Limit: Tokens & Truncation

Every model has a maximum context window (e.g. 128K or 1M tokens). Real conversations grow every turn, and every token costs money and latency — so watching the count matters long before you ever hit the hard ceiling.

In [8]:
messages = [{"role": "system", "content": "You are a helpful assistant that remembers the conversation."}]

planet_turns = [
    "Let's talk about the solar system. Tell me one striking fact about Mars.",
    "Now tell me one striking fact about Jupiter.",
    "Now tell me one striking fact about Saturn.",
]

for turn in planet_turns:
    messages.append({"role": "user", "content": turn})
    response = client.chat.completions.create(model=model, messages=messages)
    reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    print(f"tokens so far: {count_tokens(messages):>4}  |  asked: {turn}")

print("\nFull history token count:", count_tokens(messages))

tokens so far:   73  |  asked: Let's talk about the solar system. Tell me one striking fact about Mars.
tokens so far:  122  |  asked: Now tell me one striking fact about Jupiter.
tokens so far:  169  |  asked: Now tell me one striking fact about Saturn.

Full history token count: 169


👉 The context keeps growing — unbounded, that's a real bill and eventually a real wall. One common strategy is a **sliding window**: keep the system message plus only the last *N* turns, and quietly drop the rest.

In [9]:
def truncate(messages, keep_last_turns):
    system = [m for m in messages if m["role"] == "system"]
    rest = [m for m in messages if m["role"] != "system"]
    return system + rest[-keep_last_turns * 2:]


trimmed = truncate(messages, keep_last_turns=1)  # only the most recent exchange survives

print("Full history tokens:   ", count_tokens(messages))
print("Trimmed history tokens:", count_tokens(trimmed))

trimmed.append({"role": "user", "content": "What striking fact did you tell me about Mars?"})
response = client.chat.completions.create(model=model, messages=trimmed)
print("\nAnswer using only the trimmed window:\n", response.choices[0].message.content)

Full history tokens:    169
Trimmed history tokens: 57

Answer using only the trimmed window:
 I didn’t tell you about Mars yet—only about **Saturn**.


👉 Naive truncation is a blunt instrument: it shrinks tokens fast, but Mars quietly fell out of the window — the model isn't "half-remembering," it genuinely no longer has that fact anywhere in its context. Section 8 comes back with a smarter way to shrink history without losing information like this.

## 6. Grounding: Injecting Knowledge the Model Doesn't Have

An LLM only "knows" what it saw in training, plus whatever you put in its context right now. Ask about something it couldn't possibly have trained on — private, unreleased, or invented — and watch what happens with and without that information injected into the context.

In [10]:
from datetime import date


question = "Tell me the top 1 headline of today 2026-07-22?"

ungrounded = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": question}],
)
print("Without grounding:\n", ungrounded.choices[0].message.content)

Without grounding:
 I can’t determine the single “top headline of today (2026-07-22)” reliably because I don’t have live access to today’s news feed.

If you tell me **which country/source** (e.g., Reuters, BBC, NYT) or share a link/screenshot, I can extract and summarize the top headline for that publication.


In [11]:
headlines = """
Top news headlines of today 2026-07-22:
- US signs deal with Saudi Arabia that could allow kingdom to enrich nuclear fuel
- Live updates: Tropical Storm Bertha makes landfall in Louisiana, flooding hits Virginia & Carolinas
- House approves bill restricting lawmakers’ ability to purchase and sell stocks
"""

grounded = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "Answer using ONLY the news headlines below. If the answer isn't in them, say so plainly.",
        },
        {"role": "user", "content": f"Release notes:\n{headlines}\n\nQuestion: {question}"},
    ],
)
print("With the news headlines injected into context:\n", grounded.choices[0].message.content)

With the news headlines injected into context:
 US signs deal with Saudi Arabia that could allow kingdom to enrich nuclear fuel


👉 This *is* Retrieval-Augmented Generation (RAG) in miniature — no vector database required to see the core idea. Real RAG systems just automate the "find the relevant text" step (embeddings + a vector search) before doing exactly this: stuff the retrieved text into the context and ask the model to answer from it.

## 7. Few-Shot Examples: Teaching by Showing

Instructions in a system prompt describe *what* to do in words. **Few-shot examples** show the model *what the output should look like* by placing example exchanges directly in the context before the real question.

In [13]:
review = "The battery life is amazing but the camera quality is disappointing."

# zero_shot = client.chat.completions.create(
#     model=model,
#     messages=[{"role": "user", "content": f"Classify the sentiment of this review: {review}"}],
# )
# print("Zero-shot:\n", zero_shot.choices[0].message.content)

few_shot_messages = [
    {"role": "user", "content": "Classify the sentiment of this review: The screen is gorgeous and the price is fair."},
    {"role": "assistant", "content": "positive"},
    {"role": "user", "content": "Classify the sentiment of this review: It arrived broken and support never replied."},
    {"role": "assistant", "content": "negative"},
    {"role": "user", "content": "Classify the sentiment of this review: Works fine, nothing special."},
    {"role": "assistant", "content": "neutral"},
    {"role": "user", "content": f"Classify the sentiment of this review: {review}"},
]

few_shot = client.chat.completions.create(model=model, messages=few_shot_messages)
print("\nFew-shot:\n", few_shot.choices[0].message.content)


Few-shot:
 mixed


👉 Same model, same underlying task, same instructions given (none, explicitly!) — just three extra exchanges sitting in the context beforehand. That's usually enough to lock in an exact output format without writing a single line of formatting instructions.

## 8. Compacting Context: Summarize Instead of Truncate

Section 5's sliding window was cheap but lossy — old facts just vanished. A better trick for long-running conversations: ask the model itself to **compress** the history into a compact summary that preserves the facts, then keep only that summary going forward.

In [14]:
print("Original history tokens:", count_tokens(messages))

summary_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": (
                "Summarize the following conversation into a few bullet points that "
                "capture every fact mentioned, so the summary can replace the full transcript."
            ),
        },
        {
            "role": "user",
            "content": "\n".join(f"{m['role']}: {m['content']}" for m in messages if m["role"] != "system"),
        },
    ],
)
summary = summary_response.choices[0].message.content
print("\nSummary:\n", summary)

compacted_messages = [
    {"role": "system", "content": "You are a helpful assistant that remembers the conversation."},
    {"role": "assistant", "content": f"Summary of the conversation so far:\n{summary}"},
]
print("\nCompacted history tokens:", count_tokens(compacted_messages))

compacted_messages.append({"role": "user", "content": "What did we say about Mars and Jupiter?"})
response = client.chat.completions.create(model=model, messages=compacted_messages)
print("\nAnswer using the compacted context:\n", response.choices[0].message.content)

Original history tokens: 169

Summary:
 - **Mars:** Has the largest volcano in the solar system, **Olympus Mons**, about **22 km (14 miles)** high and roughly **600 km (370 miles)** wide.  
- **Jupiter:** Features the **Great Red Spot**, a storm larger than Earth that has been raging for **hundreds of years** and remains visible today.  
- **Saturn:** Has **exceptionally low density**, less dense than water—so it would **float in a hypothetical giant ocean**.

Compacted history tokens: 119

Answer using the compacted context:
 - **Mars:** We said it has the largest volcano in the solar system, **Olympus Mons**, about **22 km (14 miles)** tall and roughly **600 km (370 miles)** wide.  
- **Jupiter:** We said it has the **Great Red Spot**, a massive storm larger than Earth that has been raging for **hundreds of years** and is still visible today.


👉 Far fewer tokens than the full transcript, and — unlike the hard truncation in Section 5 — Mars survived, because summarization compresses *information density* instead of chopping off *whole turns*. This is exactly what production agents (and long-running chat products) do once a conversation grows past a comfortable size.

## Summary

| Concept | What it does |
|---|---|
| `messages` list | The entire context — the only thing the model can see |
| Statelessness | Nothing persists between calls unless you resend it |
| `system` role | Steers persona/behavior without adding facts |
| Conversation history | "Memory" is just context you chose to keep and resend |
| Tokens | The real unit of context size, cost, and the hard context-window limit |
| Sliding-window truncation | Cheap way to cap size; silently loses whatever falls out |
| Grounding / mini-RAG | Injecting facts into context beats hoping the model already knows them |
| Few-shot examples | Shapes output format/behavior by demonstration, not instruction |
| Summarization | Compacts context by density instead of by dropping whole turns |

**Practices worth keeping:**

- Treat the `messages` list as the single source of truth for what the model "knows" right now — if it's not in there, the model doesn't have it.
- Count tokens before you need to — token counts (and therefore cost and latency) creep up fast in any looping agent or chatbot.
- Prefer summarization over hard truncation once a conversation gets long; it's more code, but it doesn't silently erase facts.
- Ground answers with real retrieved/injected text whenever accuracy matters — never rely on the model "probably knowing" something.
- Reach for a couple of few-shot examples before writing paragraphs of formatting instructions — showing usually beats telling.

**Where this fits in this repo:** `tool_calling.ipynb` in this folder builds on these exact same `messages`/context mechanics to let the model request function calls; `litellm_benefits.ipynb` shows the same context primitives working unchanged across providers.